In [1]:
import pandas as pd
import polars as pl
import numpy as np
import time
import copy

C:\Users\chaya\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Creating the Data

In [27]:
np.random.seed(42)
n = 10000000

categories = ["Electronics", "Clothing", "Food", "Books", "Sports"]
regions    = ["North", "South", "East", "West"]
statuses   = ["completed", "returned", "pending"]
valid_dates = pd.date_range("2000-01-01", "2025-12-31", freq="D")

df = pd.DataFrame({
    "date":    np.random.choice(valid_dates, size=n, replace=True)
                       ,
    "product_id":    [f"P{str(i).zfill(3)}" for i in np.random.randint(1, 60, n)],
    "product":       np.random.choice(
                         ["Laptop", "Phone", "T-Shirt", "Jeans", "Rice",
                          "Bread", "Novel", "Textbook", "Tennis Racket", "Dumbbells"], n),
    "category":      np.random.choice(categories, n),
    "region":        np.random.choice(regions, n),
    "city":          np.random.choice(
                         ["Mumbai", "Delhi", "Bangalore", "Chennai",
                          "Kolkata", "Hyderabad", "Pune", "Ahmedabad"], n),
    "country":       "IN",
    "revenue":       np.round(np.random.uniform(50, 5000, n), 2),
    "quantity":      np.random.randint(1, 50, n),
    "price":         np.round(np.random.uniform(10, 1000, n), 2),
    "score":         np.random.randint(40, 100, n),
    "status":        np.random.choice(statuses, n, p=[0.7, 0.2, 0.1]),
    "is_discounted": np.random.choice([True, False], n),
    "q1":            np.random.randint(30, 100, n),
    "q2":            np.random.randint(30, 100, n),
    "q3":            np.random.randint(30, 100, n),
    "q4":            np.random.randint(30, 100, n),
})

#Randomly assigning Nulls
null_idx = np.random.choice(n, size=int(n * 0.05), replace=False)
df.loc[null_idx, "revenue"] = np.nan
df.loc[null_idx[:10], "price"] = np.nan

df.to_csv("sales.csv", index=False)
print(f"✅ sales.csv written - {n} rows, {df.shape[1]} columns")

✅ sales.csv written - 10000000 rows, 17 columns


In [28]:
df.head()

,date,product_id,product,category,region,city,country,revenue,quantity,price,score,status,is_discounted,q1,q2,q3,q4
0,2019-11-27,P006,Phone,Clothing,North,Ahmedabad,IN,3146.09,30,220.19,99,completed,True,36,60,87,93
1,2002-05-10,P057,Bread,Sports,South,Mumbai,IN,2382.93,42,599.36,49,completed,True,47,74,90,98
2,2014-10-04,P036,Novel,Books,West,Bangalore,IN,4026.06,36,354.32,43,pending,True,40,82,48,73
3,2014-03-19,P054,Phone,Clothing,East,Kolkata,IN,3311.10,38,744.53,64,pending,True,92,51,46,92
4,2015-09-13,P002,Novel,Books,North,Chennai,IN,3797.05,39,731.77,40,completed,False,99,98,56,95


In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000000 entries, 0 to 9999999
Data columns (total 17 columns):
 #   Column         Dtype         
---  ------         -----         
 0   date           datetime64[us]
 1   product_id     str           
 2   product        str           
 3   category       str           
 4   region         str           
 5   city           str           
 6   country        str           
 7   revenue        float64       
 8   quantity       int32         
 9   price          float64       
 10  score          int32         
 11  status         str           
 12  is_discounted  bool          
 13  q1             int32         
 14  q2             int32         
 15  q3             int32         
 16  q4             int32         
dtypes: bool(1), datetime64[us](1), float64(2), int32(6), str(7)
memory usage: 1.3 GB


### 1. Lazy Evaluation - Let Polars Think Before It Acts

In [30]:
# Pandas - executes line by line
# Start the timer
start_pandas = time.perf_counter()

# Pandas execution
pandas_df = pd.read_csv("sales.csv")
step_1 = pandas_df[pandas_df["region"] == "South"]
step_2 = step_1.groupby("category")["revenue"].mean()
# End the timer
end_pandas = time.perf_counter()

In [31]:
# Polars - LazyFrame: builds a plan, then executes it once
start_polars = time.perf_counter()

polars_df_1 = (
    pl.scan_csv("sales.csv")           # Nothing runs yet
    .filter(pl.col("region") == "South")
    .group_by("category")
    .agg(pl.col("revenue").mean())
    .collect()                          # Runs here - optimized
)

# End the timer
end_polars = time.perf_counter()

print(f" Pandas: {end_pandas - start_pandas} \nPolars: {end_polars - start_polars}")

 Pandas: 57.10654480010271 
Polars: 3.91276799980551


### 2. Window Functions - over()

In [32]:
# Pandas - requires transform, runs serially
start_time = time.perf_counter()

df = copy.deepcopy(pandas_df)

df["running_total"] = df.groupby("category")["revenue"].cumsum()
df["group_rank"]    = df.groupby("category")["revenue"].rank(method="dense")
df["prev_revenue"]  = df.groupby("category")["revenue"].shift(1)

end_time = time.perf_counter()

# Calculate and print elapsed time
execution_time = end_time - start_time
print(f"Pandas execution time: {execution_time:.6f} seconds")

Pandas execution time: 18.251211 seconds


In [33]:
# Polars - everything in one with_columns(), runs in parallel

polars_df = pl.read_csv("sales.csv")

start_time = time.perf_counter()
polars_df.with_columns([
    pl.col("revenue").cum_sum().over("category").alias("running_total"),
    pl.col("revenue").rank(method="dense").over("category").alias("group_rank"),
    pl.col("revenue").shift(1).over("category").alias("prev_revenue"),
])

end_time = time.perf_counter()

# Calculate and print elapsed time
execution_time = end_time - start_time
print(f"Polars execution time: {execution_time:.6f} seconds")

# Window over multiple columns
polars_df.with_columns(
    pl.col("revenue").cum_sum().over(["category", "region"]).alias("running_total")
).head()


Polars execution time: 1.149325 seconds


date,product_id,product,category,region,city,country,revenue,quantity,price,score,status,is_discounted,q1,q2,q3,q4,running_total
str,str,str,str,str,str,str,f64,i64,f64,i64,str,bool,i64,i64,i64,i64,f64
"""2019-11-27""","""P006""","""Phone""","""Clothing""","""North""","""Ahmedabad""","""IN""",3146.09,30,220.19,99,"""completed""",true,36,60,87,93,3146.09
"""2002-05-10""","""P057""","""Bread""","""Sports""","""South""","""Mumbai""","""IN""",2382.93,42,599.36,49,"""completed""",true,47,74,90,98,2382.93
"""2014-10-04""","""P036""","""Novel""","""Books""","""West""","""Bangalore""","""IN""",4026.06,36,354.32,43,"""pending""",true,40,82,48,73,4026.06
"""2014-03-19""","""P054""","""Phone""","""Clothing""","""East""","""Kolkata""","""IN""",3311.1,38,744.53,64,"""pending""",true,92,51,46,92,3311.1
"""2015-09-13""","""P002""","""Novel""","""Books""","""North""","""Chennai""","""IN""",3797.05,39,731.77,40,"""completed""",false,99,98,56,95,3797.05


### 3. Pivot & Unpivot

##### Pivot

In [34]:
# Pandas
start_time = time.perf_counter()
pandas_pivot = pandas_df.pivot_table(index="city", columns="category", values="revenue", aggfunc="mean")
end_time = time.perf_counter()

# Calculate and print elapsed time
execution_time = end_time - start_time
print(f"Pandas execution time: {execution_time:.6f} seconds")

# Polars
start_time = time.perf_counter()
polars_pivot = polars_df.pivot(index="city", columns="category", values="revenue", aggregate_function="mean")
end_time = time.perf_counter()

execution_time = end_time - start_time
print(f"Polars execution time: {execution_time:.6f} seconds")

Pandas execution time: 2.851258 seconds


C:\Users\chaya\AppData\Local\Temp\ipykernel_51744\3186819750.py:12: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  polars_pivot = polars_df.pivot(index="city", columns="category", values="revenue", aggregate_function="mean")


Polars execution time: 2.545876 seconds


In [35]:
pandas_pivot.head()

category,Books,Clothing,Electronics,Food,Sports
city,,,,,
Ahmedabad,2522.623056,2519.220592,2525.997446,2519.479276,2524.720537
Bangalore,2524.716348,2523.691884,2526.404532,2528.680926,2520.876211
Chennai,2524.133312,2525.075914,2528.547525,2526.166593,2524.828841
Delhi,2517.707847,2527.479092,2526.877778,2522.497621,2519.663444
Hyderabad,2523.957690,2526.991538,2523.941911,2521.105410,2528.970139


In [36]:
polars_pivot.head()

city,Clothing,Sports,Books,Electronics,Food
str,f64,f64,f64,f64,f64
"""Ahmedabad""",2519.220592,2524.720537,2522.623056,2525.997446,2519.479276
"""Mumbai""",2528.834391,2527.871884,2523.426442,2523.371514,2523.420394
"""Bangalore""",2523.691884,2520.876211,2524.716348,2526.404532,2528.680926
"""Kolkata""",2521.953361,2529.929667,2525.115635,2525.105753,2525.15911
"""Chennai""",2525.075914,2524.828841,2524.133312,2528.547525,2526.166593


##### Unpivot

In [37]:
# Pandas
start_time = time.perf_counter()
pandas_unpivot = pandas_pivot.reset_index().melt(id_vars=["city"], var_name="category", value_name="revenue")
end_time = time.perf_counter()
print(f"Pandas execution time: {end_time - start_time:.6f} seconds")

# Polars
start_time = time.perf_counter()
polars_unpivot = polars_pivot.unpivot(index="city", variable_name="category", value_name="revenue")
end_time = time.perf_counter()
print(f"Polars execution time: {end_time - start_time:.6f} seconds")


Pandas execution time: 0.088821 seconds
Polars execution time: 0.012571 seconds


In [38]:
pandas_unpivot.head()

,city,category,revenue
0,Ahmedabad,Books,2522.623056
1,Bangalore,Books,2524.716348
2,Chennai,Books,2524.133312
3,Delhi,Books,2517.707847
4,Hyderabad,Books,2523.957690


In [39]:
polars_unpivot.head()

city,category,revenue
str,str,f64
"""Ahmedabad""","""Clothing""",2519.220592
"""Mumbai""","""Clothing""",2528.834391
"""Bangalore""","""Clothing""",2523.691884
"""Kolkata""","""Clothing""",2521.953361
"""Chennai""","""Clothing""",2525.075914


### 4. Struct Columns - Typed Nested Data

In [40]:
# Pandas - workaround: store dicts in an object column
start_pandas = time.perf_counter()
pandas_df["location"] = pandas_df[["city", "region"]].to_dict(orient="records")
end_pandas = time.perf_counter()

start_polars = time.perf_counter()
# Polars - native Struct type
polars_df_struct = polars_df.with_columns(
    pl.struct(["city", "region"]).alias("location")
)
end_polars = time.perf_counter()

print(f" Pandas: {end_pandas - start_pandas} \nPolars: {end_polars - start_polars}")

# Access a field from the struct column
polars_df_struct.with_columns(
    pl.col("location").struct.field("city").alias("city_extracted")
).head(1) #You can then remove the individual columns

 Pandas: 146.59005449991673 
Polars: 0.12346939998678863


date,product_id,product,category,region,city,country,revenue,quantity,price,score,status,is_discounted,q1,q2,q3,q4,location,city_extracted
str,str,str,str,str,str,str,f64,i64,f64,i64,str,bool,i64,i64,i64,i64,struct[2],str
"""2019-11-27""","""P006""","""Phone""","""Clothing""","""North""","""Ahmedabad""","""IN""",3146.09,30,220.19,99,"""completed""",true,36,60,87,93,"{""Ahmedabad"",""North""}","""Ahmedabad"""


### 5. String Operations

In [41]:
# Pandas
start_pandas = time.perf_counter()
pandas_df["product_clean"] = pandas_df["product"].str.lower().str.strip()
pandas_df["first_word"]    = pandas_df["product"].str.split(" ").str[0]
pandas_df["has_t_shirt"]   = pandas_df["product"].str.contains("T-Shirt", case=False, na=False)
pandas_df["code"]          = pandas_df["product_id"].str.extract(r"(P\d{3})", expand=False)
end_pandas = time.perf_counter()

start_polars = time.perf_counter()
# Polars
polars_df_str_1 = polars_df.with_columns([
    pl.col("product").str.to_lowercase().str.strip_chars().alias("product_clean"),
    pl.col("product").str.split(" ").list.first().alias("first_word"),
    pl.col("product").str.contains("(?i)t-shirt").alias("has_t_shirt"),  # inline regex flag
    pl.col("product_id").str.extract(r"(P\d{3})", 1).alias("code"),
])
end_polars = time.perf_counter()

print(f" Pandas: {end_pandas - start_pandas} \nPolars: {end_polars - start_polars}")

 Pandas: 29.056177699938416 
Polars: 3.622880300041288


In [42]:
pandas_df.head()

,date,product_id,product,category,region,city,country,revenue,quantity,price,...,is_discounted,q1,q2,q3,q4,location,product_clean,first_word,has_t_shirt,code
0,2019-11-27,P006,Phone,Clothing,North,Ahmedabad,IN,3146.09,30,220.19,...,True,36,60,87,93,"{'city': 'Ahmedabad', 'region': 'North'}",phone,Phone,False,P006
1,2002-05-10,P057,Bread,Sports,South,Mumbai,IN,2382.93,42,599.36,...,True,47,74,90,98,"{'city': 'Mumbai', 'region': 'South'}",bread,Bread,False,P057
2,2014-10-04,P036,Novel,Books,West,Bangalore,IN,4026.06,36,354.32,...,True,40,82,48,73,"{'city': 'Bangalore', 'region': 'West'}",novel,Novel,False,P036
3,2014-03-19,P054,Phone,Clothing,East,Kolkata,IN,3311.10,38,744.53,...,True,92,51,46,92,"{'city': 'Kolkata', 'region': 'East'}",phone,Phone,False,P054
4,2015-09-13,P002,Novel,Books,North,Chennai,IN,3797.05,39,731.77,...,False,99,98,56,95,"{'city': 'Chennai', 'region': 'North'}",novel,Novel,False,P002


In [43]:
polars_df_str_1.head()

date,product_id,product,category,region,city,country,revenue,quantity,price,score,status,is_discounted,q1,q2,q3,q4,product_clean,first_word,has_t_shirt,code
str,str,str,str,str,str,str,f64,i64,f64,i64,str,bool,i64,i64,i64,i64,str,str,bool,str
"""2019-11-27""","""P006""","""Phone""","""Clothing""","""North""","""Ahmedabad""","""IN""",3146.09,30,220.19,99,"""completed""",true,36,60,87,93,"""phone""","""Phone""",false,"""P006"""
"""2002-05-10""","""P057""","""Bread""","""Sports""","""South""","""Mumbai""","""IN""",2382.93,42,599.36,49,"""completed""",true,47,74,90,98,"""bread""","""Bread""",false,"""P057"""
"""2014-10-04""","""P036""","""Novel""","""Books""","""West""","""Bangalore""","""IN""",4026.06,36,354.32,43,"""pending""",true,40,82,48,73,"""novel""","""Novel""",false,"""P036"""
"""2014-03-19""","""P054""","""Phone""","""Clothing""","""East""","""Kolkata""","""IN""",3311.1,38,744.53,64,"""pending""",true,92,51,46,92,"""phone""","""Phone""",false,"""P054"""
"""2015-09-13""","""P002""","""Novel""","""Books""","""North""","""Chennai""","""IN""",3797.05,39,731.77,40,"""completed""",false,99,98,56,95,"""novel""","""Novel""",false,"""P002"""


In [44]:
#Additional Manipulations
polars_df_str_1 = polars_df_str_1.with_columns(
    pl.col("status").str.replace_all(r"(pending|returned)", "in_review")
)

# Zero-pad product IDs - useful for joins with mixed-format keys
polars_df_str_1 = polars_df_str_1.with_columns(
    pl.col("product_id").str.zfill(5)   # "P3" → "00003"
)

# Split a compound column into two clean columns
polars_df_str_1 = polars_df_str_1.with_columns(
    pl.col("product_id").str.splitn("-", 2).alias("parts")
).unnest("parts")   # expands struct into separate columns: field_0, field_1

### 6. Datetime operations

In [45]:
# Pandas
# Convert the string column to a true datetime data type
pandas_df["date"] = pd.to_datetime(pandas_df["date"],errors="coerce") #erros is added o handle date column since 10M rows and some are out of bounds
pandas_df["year"]       = pandas_df["date"].dt.year
pandas_df["week"]       = pandas_df["date"].dt.isocalendar().week
pandas_df["next_month"] = pandas_df["date"] + pd.DateOffset(months=1)

# Polars
polars_df = polars_df.with_columns(
    pl.col("date").str.to_date("%Y-%m-%d", strict=False)
)
polars_df.with_columns([
    pl.col("date").dt.year().alias("year"),
    pl.col("date").dt.week().alias("week"),
    pl.col("date").dt.offset_by("1mo").alias("next_month"),   # clean duration string
    pl.col("date").dt.truncate("1w").alias("week_start"),     # floor to week boundary
    pl.col("date").dt.truncate("1mo").alias("month_start"),   # floor to month
])

# Resample to weekly revenue buckets
polars_df.sort("date").group_by_dynamic("date", every="1w").agg(
    pl.col("revenue").sum()
)

date,revenue
date,f64
1999-12-27,4.9657e6
2000-01-03,1.7654e7
2000-01-10,1.7474e7
2000-01-17,1.7304e7
2000-01-24,1.7901e7
…,…
2025-12-01,1.7416e7
2025-12-08,1.7843e7
2025-12-15,1.7874e7


### 7. when/then/otherwise - Deeper Dive

In [46]:
# Basic (covered in Part 1) - using pl.lit() for constant values
polars_df.with_columns(
    pl.when(pl.col("score") >= 90).then(pl.lit("A"))
    .when(pl.col("score") >= 80).then(pl.lit("B"))
    .otherwise(pl.lit("F"))
    .alias("grade")
)

# Advanced - using column expressions inside then()
polars_df.with_columns(
    pl.when(pl.col("is_discounted"))
    .then(pl.col("price") * 0.85)           # expression, not a literal
    .otherwise(pl.col("price"))
    .alias("final_price")
)

# Advanced - conditional aggregation using when inside agg()
polars_df.group_by("category").agg(
    pl.when(pl.col("status") == "returned")
    .then(pl.col("revenue"))
    .otherwise(pl.lit(0))
    .sum()
    .alias("return_revenue")
)

category,return_revenue
str,f64
"""Electronics""",9.6062e8
"""Sports""",9.5739e8
"""Books""",9.6006e8
"""Food""",9.5747e8
"""Clothing""",9.5762e8


### 8. Horizontal Aggregations - Across Columns

In [48]:
# Pandas
pandas_df["total_score"] = pandas_df[["q1", "q2", "q3", "q4"]].sum(axis=1)
pandas_df["best_score"]  = pandas_df[["q1", "q2", "q3", "q4"]].max(axis=1)
pandas_df["all_passed"]  = (pandas_df[["q1", "q2", "q3", "q4"]] >= 50).all(axis=1)

# Polars
polars_df.with_columns([
    pl.sum_horizontal("q1", "q2", "q3", "q4").alias("total_score"),
    pl.max_horizontal("q1", "q2", "q3", "q4").alias("best_score"),
    pl.all_horizontal(pl.col("q1", "q2", "q3", "q4") >= 50).alias("all_passed"),
])

import polars.selectors as cs

# Also works seamlessly with selectors
polars_df.with_columns(
    pl.sum_horizontal(cs.starts_with("q")).alias("total_score")
)

date,product_id,product,category,region,city,country,revenue,quantity,price,score,status,is_discounted,q1,q2,q3,q4,total_score
date,str,str,str,str,str,str,f64,i64,f64,i64,str,bool,i64,i64,i64,i64,i64
2019-11-27,"""P006""","""Phone""","""Clothing""","""North""","""Ahmedabad""","""IN""",3146.09,30,220.19,99,"""completed""",true,36,60,87,93,306
2002-05-10,"""P057""","""Bread""","""Sports""","""South""","""Mumbai""","""IN""",2382.93,42,599.36,49,"""completed""",true,47,74,90,98,351
2014-10-04,"""P036""","""Novel""","""Books""","""West""","""Bangalore""","""IN""",4026.06,36,354.32,43,"""pending""",true,40,82,48,73,279
2014-03-19,"""P054""","""Phone""","""Clothing""","""East""","""Kolkata""","""IN""",3311.1,38,744.53,64,"""pending""",true,92,51,46,92,319
2015-09-13,"""P002""","""Novel""","""Books""","""North""","""Chennai""","""IN""",3797.05,39,731.77,40,"""completed""",false,99,98,56,95,387
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2015-10-05,"""P050""","""Laptop""","""Books""","""East""","""Hyderabad""","""IN""",2507.93,33,22.91,87,"""returned""",true,72,63,73,67,308
2003-03-11,"""P003""","""T-Shirt""","""Electronics""","""South""","""Bangalore""","""IN""",4185.88,45,982.09,65,"""completed""",true,96,56,90,95,382
2023-09-07,"""P016""","""Laptop""","""Food""","""North""","""Mumbai""","""IN""",2876.76,3,541.98,57,"""completed""",true,71,34,35,68,211


### 9. Anti-Joins & Semi-Joins - SQL-Style Filtering Joins

In [53]:
orders_pl = polars_df.head(1000)
orders_pd = pandas_df.head(1000)
blacklist = pl.DataFrame({
    "product_id": ["P001", "P042", "P017"],
    "category": ["Electronics", "Sports", "Electronics"]
})

blacklist_pd = pd.DataFrame({
    "product_id": ["P001", "P042", "P017"],
    "category": ["Electronics", "Sports", "Electronics"]
})

vip_cats = pl.DataFrame({"category": ["Electronics", "Sports"]})


# 1. SEMI-JOIN 
#Think of semi as a way to only keep recrods in the left table that have a value in the right table, but does not pull any columns
# Pandas alternative: orders_pd[orders_pd["category"].isin(vip_cats_pd["category"])]
semi_result = orders_pl.join(vip_cats, on="category", how="semi")

# 2. ANTI-JOIN — Single Key
# Pandas alternative: orders_pd[~orders_pd["product_id"].isin(blacklist_pd["product_id"])]
anti_result = orders_pl.join(blacklist, on="product_id", how="anti")


# =====================================================================
# Why this matters over .isin(): Multi-Key Filters
# =====================================================================

# Pandas - multi-key "NOT IN" is verbose, creates row bloat, and requires dropping columns
merged = orders_pd.merge(blacklist_pd, on=["product_id", "category"], how="left", indicator=True)
orphans_pd = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

# Polars - One clean, highly memory-optimized line
orphans_pl = orders_pl.join(blacklist, on=["product_id", "category"], how="anti")

### 10. CS Selectors

In [56]:
pandas_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000000 entries, 0 to 9999999
Data columns (total 28 columns):
 #   Column         Dtype         
---  ------         -----         
 0   date           datetime64[us]
 1   product_id     str           
 2   product        str           
 3   category       str           
 4   region         str           
 5   city           str           
 6   country        str           
 7   revenue        float64       
 8   quantity       int64         
 9   price          float64       
 10  score          int64         
 11  status         str           
 12  is_discounted  bool          
 13  q1             int64         
 14  q2             int64         
 15  q3             int64         
 16  q4             int64         
 17  location       object        
 18  product_clean  str           
 19  first_word     object        
 20  has_t_shirt    bool          
 21  code           str           
 22  year           int32         
 23  week           UI

In [58]:
# Pandas - select by type, limited and non-combinable
pandas_df.select_dtypes(include="number").add_prefix("num_").head()
pandas_df.select_dtypes(include=["object", "category"]).head()

C:\Users\chaya\AppData\Local\Temp\ipykernel_51744\2605702304.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  pandas_df.select_dtypes(include=["object", "category"]).head()


,product_id,product,category,region,city,country,status,location,product_clean,first_word,code
0,P006,Phone,Clothing,North,Ahmedabad,IN,completed,"{'city': 'Ahmedabad', 'region': 'North'}",phone,Phone,P006
1,P057,Bread,Sports,South,Mumbai,IN,completed,"{'city': 'Mumbai', 'region': 'South'}",bread,Bread,P057
2,P036,Novel,Books,West,Bangalore,IN,pending,"{'city': 'Bangalore', 'region': 'West'}",novel,Novel,P036
3,P054,Phone,Clothing,East,Kolkata,IN,pending,"{'city': 'Kolkata', 'region': 'East'}",phone,Phone,P054
4,P002,Novel,Books,North,Chennai,IN,completed,"{'city': 'Chennai', 'region': 'North'}",novel,Novel,P002


In [62]:
# By type
cs.numeric()     # all int + float
cs.string()      # Utf8 columns
cs.temporal()    # date, datetime, duration
cs.boolean()

# By name pattern
cs.starts_with("q")          # q1, q2, q3, q4
cs.ends_with("_id")          # product_id, user_id
cs.contains("price")         # unit_price, base_price

cs.matches("(price)")

In [61]:

# Set operations - combine selectors freely
cs.numeric() | cs.boolean()        # numeric OR boolean
cs.numeric() & cs.starts_with("q") # numeric AND starts with q
~cs.numeric()                       # everything EXCEPT numeric
cs.numeric() - cs.starts_with("q") # numeric but NOT q-prefixed

# 2.Where selectors get powerful - inside transformations, not just select():
# Cast all float columns to Float32 in one line (memory optimisation)
polars_df.with_columns(cs.float().cast(pl.Float32))

# Round every numeric column to 2 decimal places
polars_df.with_columns(cs.numeric().round(2))

# Apply the same aggregation across all quarter columns
polars_df.group_by("category").agg(cs.starts_with("q").sum())

# Rename entire column groups by pattern
polars_df.rename({c: c.replace("q", "quarter_") for c in polars_df.select(cs.starts_with("q")).columns})

# Fill nulls across all numeric columns at once
polars_df.with_columns(cs.numeric().fill_null(0))

date,product_id,product,category,region,city,country,revenue,quantity,price,score,status,is_discounted,q1,q2,q3,q4
date,str,str,str,str,str,str,f64,i64,f64,i64,str,bool,i64,i64,i64,i64
2019-11-27,"""P006""","""Phone""","""Clothing""","""North""","""Ahmedabad""","""IN""",3146.09,30,220.19,99,"""completed""",true,36,60,87,93
2002-05-10,"""P057""","""Bread""","""Sports""","""South""","""Mumbai""","""IN""",2382.93,42,599.36,49,"""completed""",true,47,74,90,98
2014-10-04,"""P036""","""Novel""","""Books""","""West""","""Bangalore""","""IN""",4026.06,36,354.32,43,"""pending""",true,40,82,48,73
2014-03-19,"""P054""","""Phone""","""Clothing""","""East""","""Kolkata""","""IN""",3311.1,38,744.53,64,"""pending""",true,92,51,46,92
2015-09-13,"""P002""","""Novel""","""Books""","""North""","""Chennai""","""IN""",3797.05,39,731.77,40,"""completed""",false,99,98,56,95
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2015-10-05,"""P050""","""Laptop""","""Books""","""East""","""Hyderabad""","""IN""",2507.93,33,22.91,87,"""returned""",true,72,63,73,67
2003-03-11,"""P003""","""T-Shirt""","""Electronics""","""South""","""Bangalore""","""IN""",4185.88,45,982.09,65,"""completed""",true,96,56,90,95
2023-09-07,"""P016""","""Laptop""","""Food""","""North""","""Mumbai""","""IN""",2876.76,3,541.98,57,"""completed""",true,71,34,35,68
